# Taller I · Apache Spark: RDDs, DataFrames y Spark SQL

| | |
|---|---|
| **Curso** | MCDI502 · Gestión de Datos y Tecnologías Big Data |
| **Programa** | Magíster en Ciencia de Datos · Universidad Andrés Bello |
| **Evaluación** | Sumativa 2 · Taller I |
| **Grupo** | 6 |
| **Integrantes** | Fernanda Ovalle Román<br>Sebastián Cajales Cid<br>César Lorca Bacián<br>Jorge Álvarez Ossandón |
| **Repositorio** | [github.com/jaossandon/mcdi502-s2-spark](https://github.com/jaossandon/mcdi502-s2-spark) |

**Datos:** registros históricos de los Juegos Olímpicos (1896–2016) en seis archivos: deportistas (dos partes), equipos, eventos, juegos y resultados. La tabla de **resultados** es el punto de conexión que relaciona deportistas, equipos, juegos y eventos.

## Índice

| # | Sección | Indicador | Puntaje |
|---|---|---|---|
| 1 | Configuración inicial del entorno | – | 6 |
| 2 | RDDs: creación y unión | ID 3.1 | 18 |
| 3 | RDDs: transformaciones | ID 3.2 | 12 |
| 4 | DataFrames: creación e integración | ID 3.1 | 18 |
| 5 | Paralelismo | ID 3.4 | 9 |
| 6 | Inspección del DataFrame | ID 3.1 | 6 |
| 7 | Columnas calculadas | ID 3.3 | 12 |
| 8 | Agregaciones requeridas | ID 3.3 | 21 |
| 9 | Conclusiones y trazabilidad con la rúbrica | – | – |

### Cómo leer este notebook
Cada paso tiene tres partes:
1. **🎯 Qué haremos**: objetivo, decisión técnica y criterio de la rúbrica que cubre.
2. **Código**, comentado.
3. **✅ Resultado**: qué se obtuvo y cómo se interpreta.

---
# 1. Configuración inicial del entorno

Apache Spark está escrito en Scala y se ejecuta sobre la **JVM (Java Virtual Machine)**. Para usarlo desde Python se necesitan dos piezas: **Java**, que ejecuta el motor de Spark, y **PySpark**, la interfaz de Python que le envía las instrucciones.

## 1.1 Instalación de Java y Apache Spark

**🎯 Qué haremos**
- Detectar si el notebook se ejecuta en **Google Colab** o en un equipo local.
- En Colab: instalar **OpenJDK 17** y **PySpark 4.0.1** (el paquete ya incluye los binarios de Spark).
- En local: usar el entorno virtual del proyecto, donde ya están instalados.

**Decisión técnica:** se fijan versiones exactas (PySpark 4.0.1 y Java 17) para que el notebook sea **reproducible**. Spark 4.0 es la primera versión compatible con Python 3.13 y requiere Java 17 o superior.

In [1]:
# 1.1 Instalación de Java y Apache Spark
import sys, subprocess

EN_COLAB = "google.colab" in sys.modules
print("¿Ejecutando en Google Colab?:", EN_COLAB)

if EN_COLAB:
    # Java 17 sin interfaz gráfica (más liviano)
    subprocess.run("apt-get update -qq > /dev/null && "
                   "apt-get install -y -qq openjdk-17-jdk-headless > /dev/null",
                   shell=True, check=True)
    # PySpark con la misma versión que el entorno local
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyspark==4.0.1"], check=True)

# Evidencia: versiones disponibles
import pyspark
print("Versión de Python :", sys.version.split()[0])
print("Versión de PySpark:", pyspark.__version__)

¿Ejecutando en Google Colab?: False
Versión de Python : 3.11.9
Versión de PySpark: 4.0.1


**✅ Resultado**

El notebook detecta que corre en un equipo local (`False`), por lo que no reinstala nada y usa el entorno virtual del proyecto: **Python 3.13.12** y **PySpark 4.0.1**. En Colab, la misma celda instala Java 17 y PySpark 4.0.1 automáticamente, de modo que el notebook funciona en ambos entornos sin cambios.

## 1.2 Variables de entorno

**🎯 Qué haremos**

Definir las variables de entorno que Spark lee al arrancar:

| Variable | Para qué sirve |
|---|---|
| `JAVA_HOME` | Carpeta de Java que Spark usa para lanzar la JVM. |
| `SPARK_HOME` | Carpeta con los binarios de Spark (dentro del paquete `pyspark`). |
| `PYSPARK_PYTHON` | Intérprete de Python de los *workers* (procesos que ejecutan las tareas). |
| `PYSPARK_DRIVER_PYTHON` | Intérprete de Python del *driver* (proceso que coordina el trabajo). |

**Decisiones técnicas:**
- `JAVA_HOME` apunta **explícitamente a Java 17**. El equipo local también tiene Java 8, que no es compatible con Spark 4.0; sin esta variable, Spark podría tomar la versión equivocada.
- *Driver* y *workers* usan **el mismo intérprete de Python**. Si las versiones no coinciden, Spark falla al ejecutar funciones de Python sobre los RDDs.

In [2]:
# 1.2 Variables de entorno
import os, glob, platform

if EN_COLAB:
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
elif platform.system() == "Windows":
    # Busca la instalación de Java 17 (Eclipse Temurin)
    jdks = sorted(glob.glob(r"C:\Program Files\Eclipse Adoptium\jdk-17*"))
    os.environ["JAVA_HOME"] = jdks[-1]

# Java 17 primero en el PATH, para que tenga prioridad sobre Java 8
os.environ["PATH"] = os.path.join(os.environ["JAVA_HOME"], "bin") + os.pathsep + os.environ["PATH"]
os.environ["SPARK_HOME"] = os.path.dirname(pyspark.__file__)
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

# Evidencia: valores configurados
for var in ["JAVA_HOME", "SPARK_HOME", "PYSPARK_PYTHON", "PYSPARK_DRIVER_PYTHON"]:
    print(f"{var:22s} = {os.environ[var]}")

# Evidencia: versión de Java que usará Spark
java = os.path.join(os.environ["JAVA_HOME"], "bin", "java")
salida = subprocess.run([java, "-version"], capture_output=True, text=True).stderr
print("\nJava que usará Spark:", [l for l in salida.splitlines() if "version" in l][0])

JAVA_HOME              = C:\Program Files\Eclipse Adoptium\jdk-17.0.21.9-hotspot
SPARK_HOME             = c:\Users\jorge\Documents\mcdi502-s2-spark\.venv\Lib\site-packages\pyspark
PYSPARK_PYTHON         = c:\Users\jorge\Documents\mcdi502-s2-spark\.venv\Scripts\python.exe
PYSPARK_DRIVER_PYTHON  = c:\Users\jorge\Documents\mcdi502-s2-spark\.venv\Scripts\python.exe

Java que usará Spark: openjdk version "17.0.21" 2026-10-20


**✅ Resultado**

Las cuatro variables quedan definidas. `JAVA_HOME` apunta a **Java 17.0.21** (Eclipse Temurin) y no a la versión 8 que también existe en el equipo; la verificación final lo confirma (`openjdk version "17.0.21"`). `SPARK_HOME`, `PYSPARK_PYTHON` y `PYSPARK_DRIVER_PYTHON` apuntan al entorno virtual del proyecto, por lo que *driver* y *workers* usan el mismo intérprete.

## 1.3 Creación de la SparkSession y el SparkContext

**🎯 Qué haremos**

Crear los dos puntos de entrada de Spark:
- **`SparkSession` (`spark`)**: punto de entrada a **DataFrames y Spark SQL** (secciones 4 a 8).
- **`SparkContext` (`sc`)**: punto de entrada a los **RDDs** (secciones 2 y 3). Se obtiene con `spark.sparkContext`, de modo que ambos comparten la misma aplicación.

**Decisiones técnicas:**

| Configuración | Valor | Justificación |
|---|---|---|
| `master` | `local[*]` | Ejecuta Spark en este equipo usando **todos los núcleos** disponibles como *workers*. |
| `appName` | `MCDI502_S2_Grupo6` | Identifica la aplicación en la interfaz web de Spark. |
| `spark.sql.shuffle.partitions` | `8` | El valor por defecto (200) está pensado para clústeres. Con datos de este tamaño generaría cientos de tareas casi vacías tras cada `join` o `groupBy`. |
| `setLogLevel("ERROR")` | – | Oculta avisos internos de Spark para que las salidas se lean con claridad. |

In [3]:
# 1.3 SparkSession y SparkContext
from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .appName("MCDI502_S2_Grupo6")
         .master("local[*]")
         .config("spark.sql.shuffle.partitions", "8")
         .getOrCreate())

sc = spark.sparkContext          # SparkContext: punto de entrada a los RDDs
sc.setLogLevel("ERROR")

# Evidencia: la sesión está activa
print("Versión de Spark        :", spark.version)
print("Nombre de la aplicación :", sc.appName)
print("Master                  :", sc.master)
print("Paralelismo por defecto :", sc.defaultParallelism, "núcleos")
print("Particiones de shuffle  :", spark.conf.get("spark.sql.shuffle.partitions"))
print("Interfaz web de Spark   :", sc.uiWebUrl)

Versión de Spark        : 4.0.1
Nombre de la aplicación : MCDI502_S2_Grupo6
Master                  : local[*]
Paralelismo por defecto : 8 núcleos
Particiones de shuffle  : 8
Interfaz web de Spark   : http://DESKTOP-KD98F3J:4040


**✅ Resultado**

La sesión queda activa con **Spark 4.0.1** en modo `local[*]`. El paralelismo por defecto es **8**, igual al número de núcleos del equipo: Spark puede ejecutar hasta 8 tareas al mismo tiempo, una por núcleo. Las particiones de *shuffle* quedan en 8, alineadas con ese paralelismo. Desde aquí, `spark` se usa para DataFrames y SQL, y `sc` para RDDs. La interfaz web en el puerto 4040 permite monitorear los trabajos mientras se ejecutan.

## 1.4 Ubicación de los datos

**🎯 Qué haremos**

Dejar disponibles los seis archivos originales en una carpeta `data/`:
- **En local:** se usa la carpeta `data/` del repositorio, donde ya están los archivos.
- **En Colab (u otro equipo):** la carpeta no existe, así que se descargan desde el repositorio público del grupo en GitHub.

**Decisión técnica:** la entrega incluye solo el notebook y el video, no los datos. Si el notebook dependiera de subir archivos a mano, no podría ejecutarse "sin intervenciones manuales". Descargarlos desde GitHub hace que cualquier persona pueda ejecutarlo de principio a fin. La ruta se guarda en formato `/` (`as_posix`), que Spark entiende tanto en Windows como en Linux.

In [4]:
# 1.4 Ubicación de los datos
import urllib.request
from pathlib import Path

URL_BASE = "https://raw.githubusercontent.com/jaossandon/mcdi502-s2-spark/main/data/"
ARCHIVOS = ["deportista.csv", "deportista2.csv", "equipo.csv",
            "evento.csv", "resultados.csv", "juegos.json"]

# Busca data/ en la carpeta actual o en las superiores (el notebook está en notebooks/)
carpetas = [Path.cwd(), *Path.cwd().parents]
DATA_DIR = next((p / "data" for p in carpetas if (p / "data" / "deportista.csv").exists()),
                Path("data"))
DATA_DIR.mkdir(exist_ok=True)

for nombre in ARCHIVOS:
    destino = DATA_DIR / nombre
    if not destino.exists():                       # solo descarga si falta
        urllib.request.urlretrieve(URL_BASE + nombre, destino)

DATA = DATA_DIR.resolve().as_posix()               # ruta que usará Spark

# Evidencia: archivos disponibles
print("Carpeta de datos:", DATA, "\n")
print(f"{'Archivo':18s}{'Tamaño (bytes)':>16s}")
for nombre in ARCHIVOS:
    print(f"{nombre:18s}{(DATA_DIR / nombre).stat().st_size:>16,d}")

Carpeta de datos: C:/Users/jorge/Documents/mcdi502-s2-spark/data 

Archivo             Tamaño (bytes)
deportista.csv           2,832,265
deportista2.csv          2,874,875
equipo.csv                  23,606
evento.csv                  35,144
resultados.csv           6,443,911
juegos.json                  5,232


**✅ Resultado**

Los seis archivos están disponibles en la carpeta `data/` del repositorio:

| Archivo | Tamaño | Contenido |
|---|---|---|
| `deportista.csv` | 2,8 MB | deportistas con id 1 al 67.786 |
| `deportista2.csv` | 2,9 MB | deportistas con id 67.787 al 135.571 |
| `equipo.csv` | 24 KB | 1.184 equipos |
| `evento.csv` | 35 KB | 765 eventos |
| `resultados.csv` | 6,4 MB | 271.116 participaciones |
| `juegos.json` | 5 KB | 51 ediciones de los Juegos |

**Cierre de la sección 1:** Java 17 y PySpark 4.0.1 operativos, variables de entorno definidas, `SparkSession` y `SparkContext` activos y datos disponibles. Todo se ejecuta sin intervención manual, tanto en local como en Colab.

## 2. RDDs: creación y unión (ID 3.1 · 18 pts)

Un **RDD** (*Resilient Distributed Dataset*) es la estructura básica de Spark: una colección de elementos **inmutable**, **distribuida en particiones** y **tolerante a fallos** (si una partición se pierde, Spark la reconstruye con su linaje de transformaciones).

### 2.1 RDD `deportista` con 6 particiones

🎯 **Qué haremos**
- Leer `deportista.csv` como texto con `sc.textFile()`. Cada elemento del RDD será **una línea** del archivo.
- Pedir **6 particiones** con `minPartitions=6`, tal como exige la pauta. Cada partición es un bloque de datos que un núcleo procesa en paralelo.
- Verificar el número de particiones con `getNumPartitions()` y mirar las primeras líneas con `take(3)`.

**Decisión técnica:** el archivo **no trae encabezado**, así que no hay que eliminar una fila de títulos antes de unir y contar.

In [5]:
# 2.1 Crear el RDD deportista con 6 particiones
deportista = sc.textFile(f"{DATA}/deportista.csv", minPartitions=6)

print("Tipo de objeto          :", type(deportista).__name__)
print("Número de particiones   :", deportista.getNumPartitions())
print("Primeras 3 líneas (texto):")
for linea in deportista.take(3):
    print("  ", linea)

Tipo de objeto          : RDD
Número de particiones   : 6
Primeras 3 líneas (texto):
   1,A Dijiang,1,24,180,80,199
   2,A Lamusi,1,23,170,60,199
   3,Gunnar Nielsen Aaby,1,24,0,0,273


✅ **Resultado**
- Se creó el RDD `deportista` con **6 particiones**, como pide la pauta (`getNumPartitions() = 6`).
- Cada elemento es **una línea de texto** (`'1,A Dijiang,1,24,180,80,199'`): el RDD todavía no conoce columnas ni tipos. Eso lo resolveremos al convertirlo a DataFrame.
- `textFile` es una operación **perezosa (lazy)**: Spark solo leyó el archivo cuando ejecutamos la **acción** `take(3)`.
- En la tercera línea aparecen `0,0` en altura y peso. Un deportista no puede medir 0 cm, así que trataremos esos ceros como **valores faltantes (null)**.

In [6]:
# 2.2 RDD deportista2, unión y conteo
deportista2 = sc.textFile(f"{DATA}/deportista2.csv")

deportistaTotal = deportista.union(deportista2)

n1, n2, nt = deportista.count(), deportista2.count(), deportistaTotal.count()
print(f"{'RDD':16s} {'Registros':>10s} {'Particiones':>12s}")
print(f"{'deportista':16s} {n1:>10,d} {deportista.getNumPartitions():>12d}")
print(f"{'deportista2':16s} {n2:>10,d} {deportista2.getNumPartitions():>12d}")
print(f"{'deportistaTotal':16s} {nt:>10,d} {deportistaTotal.getNumPartitions():>12d}")
print("\n¿La unión conserva todos los registros?:", nt == n1 + n2)
print("Primera línea de deportista2:", deportista2.first())

RDD               Registros  Particiones
deportista           67,786            6
deportista2          67,785            2
deportistaTotal     135,571            8

¿La unión conserva todos los registros?: True
Primera línea de deportista2: 67787,Lee BongJu,1,27,167,56,970


✅ **Resultado**

| RDD | Registros | Particiones |
|---|---:|---:|
| `deportista` | 67.786 | 6 |
| `deportista2` | 67.785 | 2 |
| `deportistaTotal` | **135.571** | **8** |

- La unión conserva **todos** los registros: 67.786 + 67.785 = 135.571 (`True`). No se pierde ni se duplica ninguna fila.
- `deportista2` empieza en el id **67.787**, justo donde termina `deportista` (67.786). Los archivos son **dos mitades contiguas** de la misma tabla, sin ids repetidos.
- `deportistaTotal` tiene **6 + 2 = 8 particiones**: `union()` solo concatena las particiones, sin *shuffle*, lo que la hace una operación barata.
- `deportista2` quedó con 2 particiones porque no le pedimos un número; Spark usa por defecto `min(núcleos, 2)`.

### 2.3 Parseo del RDD: de líneas de texto a registros con tipo

🎯 **Qué haremos**
- Definir una función `parsear()` que separa cada línea por comas y convierte cada campo a su tipo: `deportista_id`, `genero`, `edad` y `equipo_id` a **entero**; `altura` y `peso` a **decimal**; `nombre` como texto.
- Aplicarla a todo `deportistaTotal` con la transformación `map()`, obteniendo el RDD `deportistaRDD` de **tuplas**.

**Decisiones técnicas**
1. **Ceros como nulos:** en `edad`, `altura` y `peso` un valor `0` significa "dato no registrado" (nadie mide 0 cm). Los convertimos a `None` para que no distorsionen promedios ni mínimos.
2. **Cortar a 7 campos (`[:7]`):** la fila del deportista **16587** trae una coma extra al final, lo que genera una 8.ª columna vacía. Tomar solo los primeros 7 campos lo resuelve sin perder información.

In [7]:
# 2.3 Parsear cada línea a una tupla con tipos
def a_nulo(valor, tipo):
    """Convierte el texto al tipo indicado; '0' o vacío se consideran dato faltante (None)."""
    valor = valor.strip()
    return None if valor in ("", "0") else tipo(valor)

def parsear(linea):
    c = linea.split(",")[:7]          # solo 7 campos (la fila 16587 trae una coma extra)
    return (int(c[0]),                # deportista_id
            c[1].strip(),             # nombre
            int(c[2]),                # genero (1 = hombre, 2 = mujer)
            a_nulo(c[3], int),        # edad
            a_nulo(c[4], float),      # altura (cm)
            a_nulo(c[5], float),      # peso (kg)
            int(c[6]))                # equipo_id

deportistaRDD = deportistaTotal.map(parsear)

print("Fila 16587 en bruto :", deportistaTotal.filter(lambda l: l.startswith("16587,")).first())
print("Fila 16587 parseada :", deportistaRDD.filter(lambda r: r[0] == 16587).first())
print("\nPrimeras 3 tuplas:")
for t in deportistaRDD.take(3):
    print("  ", t)
print("\nRegistros parseados:", f"{deportistaRDD.count():,}")

Fila 16587 en bruto : 16587,Vladimr Bil,1,24,191,92,67,
Fila 16587 parseada : (16587, 'Vladimr Bil', 1, 24, 191.0, 92.0, 67)

Primeras 3 tuplas:
   (1, 'A Dijiang', 1, 24, 180.0, 80.0, 199)
   (2, 'A Lamusi', 1, 23, 170.0, 60.0, 199)
   (3, 'Gunnar Nielsen Aaby', 1, 24, None, None, 273)

Registros parseados: 135,571


✅ **Resultado**
- Las **135.571** líneas se parsearon sin errores a tuplas de 7 campos con sus tipos (`int`, `str`, `float`).
- **Fila 16587:** en bruto termina en una coma extra (`...,92,67,`). Tras el corte `[:7]` queda con sus 7 valores correctos (`equipo_id = 67`), sin perder datos.
- **Ceros a nulos:** el deportista 3 (*Gunnar Nielsen Aaby*) tenía `0,0` en altura y peso, y ahora tiene `None, None`. Así esos ceros no bajarán artificialmente los promedios ni aparecerán como mínimos.
- `map()` es una **transformación**: no se ejecutó nada hasta las acciones `first()`, `take()` y `count()`.

### 2.4 Conversión del RDD a DataFrame `deportista`

🎯 **Qué haremos**
- Definir un **esquema explícito** con `StructType` que nombra las 7 columnas y fija su tipo.
- Crear el DataFrame `deportista` con `spark.createDataFrame(deportistaRDD, esquema)`.
- Mostrar las primeras filas, el esquema y la cantidad de **nulos por columna**, para documentar el efecto de la decisión "ceros como nulos".

**Decisión técnica:** con un esquema explícito, Spark **no infiere tipos** (eso lo obligaría a recorrer los datos) y garantiza que `edad` sea entera y `altura`/`peso` decimales. Usamos el nombre `deportista` que pide la pauta. Desde aquí, `deportista` pasa a ser el **DataFrame**; el RDD original sigue disponible dentro de `deportistaTotal`.

In [8]:
# 2.4 Convertir el RDD a DataFrame con esquema explícito
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType

esquema_deportista = StructType([
    StructField("deportista_id", IntegerType(), False),
    StructField("nombre",        StringType(),  True),
    StructField("genero",        IntegerType(), True),
    StructField("edad",          IntegerType(), True),
    StructField("altura",        DoubleType(),  True),
    StructField("peso",          DoubleType(),  True),
    StructField("equipo_id",     IntegerType(), True),
])

deportista = spark.createDataFrame(deportistaRDD, esquema_deportista)

deportista.show(5)
deportista.printSchema()
print(f"Filas del DataFrame deportista: {deportista.count():,}")

print("\nNulos por columna:")
deportista.select([F.sum(F.col(c).isNull().cast("int")).alias(c)
                   for c in deportista.columns]).show()

+-------------+--------------------+------+----+------+----+---------+
|deportista_id|              nombre|genero|edad|altura|peso|equipo_id|
+-------------+--------------------+------+----+------+----+---------+
|            1|           A Dijiang|     1|  24| 180.0|80.0|      199|
|            2|            A Lamusi|     1|  23| 170.0|60.0|      199|
|            3| Gunnar Nielsen Aaby|     1|  24|  NULL|NULL|      273|
|            4|Edgar Lindenau Aabye|     1|  34|  NULL|NULL|      278|
|            5|Christine Jacoba ...|     2|  21| 185.0|82.0|      705|
+-------------+--------------------+------+----+------+----+---------+
only showing top 5 rows
root
 |-- deportista_id: integer (nullable = false)
 |-- nombre: string (nullable = true)
 |-- genero: integer (nullable = true)
 |-- edad: integer (nullable = true)
 |-- altura: double (nullable = true)
 |-- peso: double (nullable = true)
 |-- equipo_id: integer (nullable = true)

Filas del DataFrame deportista: 135,571

Nulos por col

✅ **Resultado**
- Se creó el **DataFrame `deportista`** con las 7 columnas que pide la pauta y **135.571 filas**, las mismas del RDD `deportistaTotal`.
- El esquema confirma los tipos: `deportista_id`, `genero`, `edad` y `equipo_id` como **integer**; `altura` y `peso` como **double**; `nombre` como **string**.
- **Nulos por columna** (efecto de tratar los ceros como dato faltante):

| Columna | Nulos | % del total |
|---|---:|---:|
| edad | 6.368 | 4,7 % |
| altura | 33.916 | 25,0 % |
| peso | 34.885 | 25,7 % |
| resto | 0 | 0 % |

- Uno de cada cuatro deportistas no tiene altura ni peso registrados, sobre todo en los Juegos más antiguos. Si los hubiéramos dejado como `0`, el promedio de altura habría caído artificialmente y el mínimo sería 0 cm. Esto será importante en el **IMC** (sección 7) y en las **agregaciones** (sección 8).

**Cierre de la sección 2:** creamos el RDD `deportista` con **6 particiones**, el RDD `deportista2`, los unimos en `deportistaTotal` (**135.571** registros, 8 particiones) y lo convertimos al DataFrame `deportista` con un esquema explícito.

## 3. RDDs: transformaciones (ID 3.2 · 12 pts)

Las **transformaciones** (`map`, `filter`, ...) crean un RDD nuevo a partir de otro, sin modificar el original (los RDDs son inmutables), y son **perezosas**: solo se ejecutan cuando llega una **acción** (`count`, `take`, `collect`...).

### 3.1 RDD `MayorEdad`: deportistas de 18 años o más

🎯 **Qué haremos**
- Aplicar `filter()` sobre `deportistaRDD` (las tuplas tipadas de la sección 2) para quedarnos con los deportistas con **edad ≥ 18**.
- Mostrar el **efecto** del filtro: registros antes y después, y verificar que la edad mínima del nuevo RDD sea 18.

**Decisión técnica:** filtramos sobre `deportistaRDD` y no sobre el texto, porque la edad ya es un **entero** (comparar `"9" >= "18"` como texto daría un resultado incorrecto). Los deportistas **sin edad registrada** (`None`) quedan fuera: no sabemos si son mayores de edad.

In [9]:
# 3.1 RDD MayorEdad: edad >= 18
EDAD = 3   # posición de la edad en la tupla

MayorEdad = deportistaRDD.filter(lambda r: r[EDAD] is not None and r[EDAD] >= 18)

total     = deportistaRDD.count()
mayores   = MayorEdad.count()
menores   = deportistaRDD.filter(lambda r: r[EDAD] is not None and r[EDAD] < 18).count()
sin_edad  = deportistaRDD.filter(lambda r: r[EDAD] is None).count()

print(f"Deportistas en total      : {total:>9,d}")
print(f"  MayorEdad (edad >= 18)  : {mayores:>9,d}  ({mayores/total:.1%})")
print(f"  Menores de 18           : {menores:>9,d}")
print(f"  Sin edad registrada     : {sin_edad:>9,d}")
print(f"  ¿Suman el total?        : {mayores + menores + sin_edad == total}")
print(f"\nEdad mínima en MayorEdad  : {MayorEdad.map(lambda r: r[EDAD]).min()}")
print(f"Edad máxima en MayorEdad  : {MayorEdad.map(lambda r: r[EDAD]).max()}")
print("\nPrimeros 3 registros de MayorEdad:")
for r in MayorEdad.take(3):
    print("  ", r)

Deportistas en total      :   135,571
  MayorEdad (edad >= 18)  :   123,244  (90.9%)
  Menores de 18           :     5,959
  Sin edad registrada     :     6,368
  ¿Suman el total?        : True

Edad mínima en MayorEdad  : 18
Edad máxima en MayorEdad  : 97

Primeros 3 registros de MayorEdad:
   (1, 'A Dijiang', 1, 24, 180.0, 80.0, 199)
   (2, 'A Lamusi', 1, 23, 170.0, 60.0, 199)
   (3, 'Gunnar Nielsen Aaby', 1, 24, None, None, 273)


✅ **Resultado**
- `MayorEdad` contiene **123.244** deportistas (**90,9 %** del total), todos con edad entre **18 y 97** años. La edad mínima es 18, así que el filtro funcionó.
- El total se descompone sin pérdidas: **123.244** mayores + **5.959** menores + **6.368** sin edad = **135.571** (`True`).
- Los 6.368 registros sin edad quedan fuera del filtro, porque no podemos afirmar que sean mayores de edad. Esa es la consecuencia directa de haber tratado los ceros como nulos.
- La edad máxima de 97 años parece un error, pero corresponde a competidores de las antiguas competencias de **arte** olímpicas, que se otorgaban por obras y no por desempeño físico.

### 3.2 RDD `Deportistas_mujer`: deportistas con `genero = 2`

🎯 **Qué haremos**
- Filtrar `deportistaRDD` con `filter()` para quedarnos con las deportistas de **género 2 (mujer)**.
- Mostrar el **efecto**: distribución de género antes del filtro (con `map` + `reduceByKey`, el patrón clave-valor típico de RDDs) y comprobar que después del filtro **solo queda el género 2**.

**Decisión técnica:** `reduceByKey` suma los conteos **dentro de cada partición antes de mover datos** entre ellas (combinación local), por lo que es más eficiente que agrupar todos los registros y contarlos después (`groupByKey`).

In [10]:
# 3.2 RDD Deportistas_mujer: genero == 2
GENERO = 2   # posición del género en la tupla

Deportistas_mujer = deportistaRDD.filter(lambda r: r[GENERO] == 2)

# Distribución de género ANTES del filtro (patrón clave-valor)
dist_genero = (deportistaRDD.map(lambda r: (r[GENERO], 1))
                            .reduceByKey(lambda a, b: a + b)
                            .sortByKey()
                            .collect())
print("Distribución de género antes del filtro (genero, cantidad):", dist_genero)

n_mujeres = Deportistas_mujer.count()
print(f"\nDeportistas_mujer          : {n_mujeres:,} ({n_mujeres/deportistaRDD.count():.1%} del total)")
print("Géneros presentes después  :", Deportistas_mujer.map(lambda r: r[GENERO]).distinct().collect())
print("\nPrimeros 3 registros de Deportistas_mujer:")
for r in Deportistas_mujer.take(3):
    print("  ", r)

Distribución de género antes del filtro (genero, cantidad): [(1, 101590), (2, 33981)]

Deportistas_mujer          : 33,981 (25.1% del total)
Géneros presentes después  : [2]

Primeros 3 registros de Deportistas_mujer:
   (5, 'Christine Jacoba Aaftink', 2, 21, 185.0, 82.0, 705)
   (8, 'Cornelia Cor Aalten Strannood', 2, 18, 168.0, None, 705)
   (13, 'Minna Maarit Aalto', 2, 30, 159.0, 55.5, 350)


✅ **Resultado**
- Antes del filtro hay **101.590 hombres** (género 1) y **33.981 mujeres** (género 2), que suman los 135.571 deportistas.
- `Deportistas_mujer` contiene **33.981** registros (**25,1 %** del total). Después del filtro el único género presente es `[2]`, así que el filtro es correcto.
- La proporción refleja la **historia de los Juegos**: las mujeres no compitieron en 1896 y su participación creció gradualmente durante el siglo XX.
- Los nulos de altura y peso **no siempre coinciden**: *Cornelia Cor Aalten Strannood* tiene altura (168) pero no peso. Lo tendremos en cuenta al calcular el IMC, que necesita ambos datos.

### 3.3 Transformación a mayúsculas de `deportistaTotal`

🎯 **Qué haremos**
- Aplicar `map()` con `str.upper()` sobre **cada línea** de `deportistaTotal` para pasar todo su contenido a **mayúsculas**, en el nuevo RDD `deportistaMayus`.
- Mostrar el **efecto**: las mismas líneas antes y después, verificar que la cantidad de registros no cambia y que no queda **ninguna letra minúscula**.

**Decisión técnica:** usamos `map()` porque es una transformación **1 a 1** (cada línea produce exactamente una línea), sin *shuffle* y conservando las mismas 8 particiones. Trabajamos sobre `deportistaTotal` (el texto), como pide la pauta, ya que solo los nombres tienen letras; los números no cambian.

In [11]:
# 3.3 Pasar a mayúsculas todo el RDD deportistaTotal
deportistaMayus = deportistaTotal.map(lambda linea: linea.upper())

print("Antes (deportistaTotal)            ->  Después (deportistaMayus)")
for antes, despues in zip(deportistaTotal.take(4), deportistaMayus.take(4)):
    print(f"  {antes:34s} ->  {despues}")

print(f"\nRegistros antes / después : {deportistaTotal.count():,} / {deportistaMayus.count():,}")
print(f"Particiones antes/después : {deportistaTotal.getNumPartitions()} / {deportistaMayus.getNumPartitions()}")
print(f"Líneas con minúsculas antes  : {deportistaTotal.filter(lambda l: l != l.upper()).count():,}")
print(f"Líneas con minúsculas después: {deportistaMayus.filter(lambda l: l != l.upper()).count():,}")

Antes (deportistaTotal)            ->  Después (deportistaMayus)
  1,A Dijiang,1,24,180,80,199        ->  1,A DIJIANG,1,24,180,80,199
  2,A Lamusi,1,23,170,60,199         ->  2,A LAMUSI,1,23,170,60,199
  3,Gunnar Nielsen Aaby,1,24,0,0,273 ->  3,GUNNAR NIELSEN AABY,1,24,0,0,273
  4,Edgar Lindenau Aabye,1,34,0,0,278 ->  4,EDGAR LINDENAU AABYE,1,34,0,0,278

Registros antes / después : 135,571 / 135,571
Particiones antes/después : 8 / 8
Líneas con minúsculas antes  : 135,571
Líneas con minúsculas después: 0


✅ **Resultado**
- Todas las líneas pasaron a mayúsculas: antes **135.571** líneas tenían minúsculas (todas incluyen un nombre) y después **0**.
- La cantidad de registros (**135.571**) y de particiones (**8**) no cambia: `map()` es una transformación 1 a 1 y **sin *shuffle***.
- Solo cambian los nombres (`A Dijiang` → `A DIJIANG`); los valores numéricos quedan intactos.
- `deportistaTotal` **no se modificó**: `map()` creó un RDD nuevo (`deportistaMayus`), porque los RDDs son **inmutables**.

**Cierre de la sección 3**

| RDD | Transformación | Registros | Efecto verificado |
|---|---|---:|---|
| `MayorEdad` | `filter(edad ≥ 18)` | 123.244 | edad mínima = 18 |
| `Deportistas_mujer` | `filter(genero == 2)` | 33.981 | único género presente = 2 |
| `deportistaMayus` | `map(upper)` | 135.571 | 0 líneas con minúsculas |

Las tres son **transformaciones perezosas**: Spark solo construye el plan (linaje) y lo ejecuta cuando llega una **acción** como `count()` o `take()`.

## 4. DataFrames: creación, optimización e integración (ID 3.1 · 18 pts)

En esta sección creamos un DataFrame por cada tabla y luego los integramos en uno solo. Las **optimizaciones** aplicadas son:
1. **Esquemas explícitos con tipos correctos** (enteros para ids): Spark no tiene que inferir tipos y los joins comparan números, no textos.
2. **Nombres de columnas homogéneos** (`evento_id`, `equipo_id`, `juego_id`) para que los joins sean directos.
3. **`broadcast`** de las tablas pequeñas al integrar (se explica en 4.5).
4. **`cache`** del DataFrame integrado, que se reutiliza en todas las secciones siguientes.

### 4.1 DataFrame `evento`

🎯 **Qué haremos**
- Leer `evento.csv` como **texto** (RDD), quitar el encabezado y parsear cada línea con el módulo `csv` de Python.
- Convertirlo a DataFrame con esquema `evento_id INT, evento STRING, deporte_id INT`.

**Decisión técnica:** 280 de las 765 líneas vienen **envueltas completas entre comillas** (`"40,""Speed Skating Men's 1,500 metres"",5"`). Un lector CSV estándar las tomaría como **una sola columna**. Por eso quitamos las comillas externas, cambiamos `""` por `"` y recién entonces separamos los campos. Así también respetamos las comas **dentro** del nombre del evento (`1,500 metres`). Además, 2 eventos tienen `deporte_id = "#N/A"`, que convertimos a nulo.

In [12]:
# 4.1 DataFrame evento (lectura como texto + parseo manual)
import csv

def parsear_evento(linea):
    l = linea.strip()
    if l.startswith('"') and l.endswith('"'):      # línea envuelta completa en comillas
        l = l[1:-1].replace('""', '"')
    c = next(csv.reader([l]))                       # respeta comas dentro de comillas
    return (int(c[0]), c[1], None if c[2] == "#N/A" else int(c[2]))

evento_txt = sc.textFile(f"{DATA}/evento.csv")
encabezado = evento_txt.first()
print("Encabezado original       :", encabezado)
print("Líneas envueltas en comillas:", evento_txt.filter(lambda l: l.startswith('"')).count())

evento = spark.createDataFrame(
    evento_txt.filter(lambda l: l != encabezado).map(parsear_evento),
    "evento_id INT, evento STRING, deporte_id INT")

print(f"Filas del DataFrame evento: {evento.count():,}\n")
evento.show(5, truncate=False)
print("Casos especiales (comillas, comas internas y #N/A):")
evento.filter("evento_id IN (30, 40) OR deporte_id IS NULL").show(truncate=False)
evento.printSchema()

Encabezado original       : cod_evento,evento,deporte_id
Líneas envueltas en comillas: 280
Filas del DataFrame evento: 765

+---------+--------------------------------+----------+
|evento_id|evento                          |deporte_id|
+---------+--------------------------------+----------+
|1        |Basketball Men's Basketball     |1         |
|2        |Judo Men's Extra-Lightweight    |2         |
|3        |Football Men's Football         |3         |
|4        |Tug-Of-War Men's Tug-Of-War     |4         |
|5        |Speed Skating Women's 500 metres|5         |
+---------+--------------------------------+----------+
only showing top 5 rows
Casos especiales (comillas, comas internas y #N/A):
+---------+-------------------------------------------------+----------+
|evento_id|evento                                           |deporte_id|
+---------+-------------------------------------------------+----------+
|30       |Art Competitions Mixed Sculpturing, Unknown Event|14        |
|40 

✅ **Resultado**
- El DataFrame `evento` tiene **765 filas**, una por cada línea de datos del archivo, sin perder ninguna.
- Las **280 líneas envueltas en comillas** se parsearon correctamente: el evento 40 conserva la coma de `1,500 metres` y el 30 la de `Sculpturing, Unknown Event`, sin desplazar columnas.
- Los 2 eventos con `deporte_id = "#N/A"` (703 *Sailing* y 743 *Motorboating*) quedaron como `NULL`, en vez de romper la conversión a entero.
- Renombramos `cod_evento` → **`evento_id`**, el mismo nombre que usa la tabla de resultados, para facilitar el join.

### 4.2 DataFrame `equipo`

🎯 **Qué haremos**
- Leer `equipo.csv` con `spark.read.csv`, con encabezado y **esquema explícito** `equipo_id INT, equipo STRING, sigla STRING`.
- Mostrar las primeras filas, el conteo y verificar un caso con comillas internas.

**Decisiones técnicas**
- **Esquema explícito en lugar de `inferSchema`:** evita una pasada extra sobre el archivo para adivinar tipos y renombra `id` → **`equipo_id`**, el nombre que usa la tabla de deportistas.
- **`escape='"'`:** algunos nombres contienen comillas escritas como `""` (estándar CSV). Spark usa `\` como escape por defecto, así que sin esta opción el nombre quedaría como `"Large boat; ... ""Poros""-1"`.

In [13]:
# 4.2 DataFrame equipo
equipo = spark.read.csv(f"{DATA}/equipo.csv", header=True, escape='"',
                        schema="equipo_id INT, equipo STRING, sigla STRING")

print(f"Filas del DataFrame equipo: {equipo.count():,}")
print(f"Siglas (países/comités) distintas: {equipo.select('sigla').distinct().count()}\n")
equipo.show(5)
print("Caso con comillas internas y punto y coma en el nombre:")
equipo.filter("equipo_id IN (79, 565)").show(truncate=False)
equipo.printSchema()

Filas del DataFrame equipo: 1,184
Siglas (países/comités) distintas: 230

+---------+--------------------+-----+
|equipo_id|              equipo|sigla|
+---------+--------------------+-----+
|        1|         30. Februar|  AUT|
|        2|A North American ...|  MEX|
|        3|           Acipactli|  MEX|
|        4|             Acturus|  ARG|
|        5|         Afghanistan|  AFG|
+---------+--------------------+-----+
only showing top 5 rows
Caso con comillas internas y punto y coma en el nombre:
+---------+-----------------------------------------------+-----+
|equipo_id|equipo                                         |sigla|
+---------+-----------------------------------------------+-----+
|79       |Bagatelle Polo Club; Paris                     |FRA  |
|565      |Large boat; Central Naval Prep School "Poros"-1|GRE  |
+---------+-----------------------------------------------+-----+

root
 |-- equipo_id: integer (nullable = true)
 |-- equipo: string (nullable = true)
 |-- sigla: s

✅ **Resultado**
- El DataFrame `equipo` tiene **1.184 equipos** que representan a **230 siglas** (países o comités olímpicos). Un mismo país puede tener varios equipos, por ejemplo clubes o botes en los primeros Juegos.
- Con `escape='"'`, el equipo 565 queda como `Large boat; Central Naval Prep School "Poros"-1`, sin comillas duplicadas. El `;` dentro del nombre (equipo 79) tampoco causa problemas porque el separador es la coma.
- Los tipos son los correctos (`equipo_id` entero) y la columna se renombró de `id` a **`equipo_id`** para el join con `deportista`.

### 4.3 DataFrame `resultado`

🎯 **Qué haremos**
- Leer `resultados.csv` con separador **`;`**, encabezado y esquema explícito.
- Limpiar dos columnas:
  - `medalla`: el texto `"NA"` (sin medalla) pasa a **nulo** real.
  - `evento_id`: se lee como texto y se convierte a entero con **`try_cast`**, porque 9 filas traen `"#N/A"`.
- Mostrar el conteo, la distribución de medallas y el esquema final.

**Decisiones técnicas**
- **`"NA"` → nulo:** si la dejáramos como texto, `"NA"` aparecería como una "cuarta medalla" en las agregaciones, y `count("medalla")` contaría filas sin medalla.
- **`try_cast` en vez de `cast`:** Spark 4 trabaja en **modo ANSI** (estricto), donde convertir `"#N/A"` a entero produce un error. `try_cast` devuelve nulo en esos casos y mantiene el resto de los valores.

In [14]:
# 4.3 DataFrame resultado
resultado_raw = spark.read.csv(f"{DATA}/resultados.csv", sep=";", header=True,
    schema="resultado_id INT, medalla STRING, deportista_id INT, juego_id INT, evento_id STRING")

print("Antes de limpiar:")
print("  medalla = 'NA'     :", f"{resultado_raw.filter(F.col('medalla') == 'NA').count():,}")
print("  evento_id = '#N/A' :", resultado_raw.filter(F.col("evento_id") == "#N/A").count())

resultado = (resultado_raw
    .withColumn("medalla", F.when(F.col("medalla") == "NA", None).otherwise(F.col("medalla")))
    .withColumn("evento_id", F.expr("try_cast(evento_id AS INT)")))

print(f"\nFilas del DataFrame resultado: {resultado.count():,}")
print("evento_id nulos tras try_cast:", resultado.filter(F.col("evento_id").isNull()).count(), "\n")
resultado.show(5)
print("Distribución de medallas (NULL = participó sin medalla):")
resultado.groupBy("medalla").count().orderBy(F.desc("count")).show()
resultado.printSchema()

Antes de limpiar:
  medalla = 'NA'     : 231,333
  evento_id = '#N/A' : 9

Filas del DataFrame resultado: 271,116
evento_id nulos tras try_cast: 9 

+------------+-------+-------------+--------+---------+
|resultado_id|medalla|deportista_id|juego_id|evento_id|
+------------+-------+-------------+--------+---------+
|           1|   NULL|            1|      39|        1|
|           2|   NULL|            2|      49|        2|
|           3|   NULL|            3|       7|        3|
|           4|   Gold|            4|       2|        4|
|           5|   NULL|            5|      36|        5|
+------------+-------+-------------+--------+---------+
only showing top 5 rows
Distribución de medallas (NULL = participó sin medalla):
+-------+------+
|medalla| count|
+-------+------+
|   NULL|231333|
|   Gold| 13372|
| Bronze| 13295|
| Silver| 13116|
+-------+------+

root
 |-- resultado_id: integer (nullable = true)
 |-- medalla: string (nullable = true)
 |-- deportista_id: integer (nullable = 

✅ **Resultado**
- El DataFrame `resultado` tiene **271.116 filas**: cada una es la participación de un deportista en un evento de unos Juegos.
- **Medallas:** 231.333 participaciones (**85,3 %**) no obtuvieron medalla y ahora son `NULL` real, no el texto `"NA"`. Las medallas se reparten casi en partes iguales: **13.372 oros**, **13.295 bronces** y **13.116 platas** (39.783 en total).
- Los **9** `evento_id = "#N/A"` quedaron como `NULL` gracias a `try_cast`, sin detener la ejecución por el modo ANSI. El resto de los ids son enteros válidos.
- En el esquema final todos los ids son **integer**, el tipo necesario para los joins con `deportista`, `evento` y `juego`.

### 4.4 DataFrame `juego`

🎯 **Qué haremos**
- Leer `juegos.json` con `spark.read.json` y la opción **`multiLine=True`**.
- Seleccionar y **renombrar** las columnas según su contenido real, con los tipos correctos.

**Decisiones técnicas**
- **`multiLine=True`:** el archivo es un único arreglo JSON repartido en varias líneas. Sin esta opción, Spark lo leería como registros corruptos (`_corrupt_record`).
- **Renombrar columnas:** los nombres originales no corresponden a su contenido:

| Columna original | Contenido real | Nuevo nombre |
|---|---|---|
| `ano` | `"1896 Verano"` | `nombre_juego` |
| `temporada` | `1896` | `anio` |
| `ciudad` | `"Verano"` / `"Invierno"` | `temporada` |

- **Tipos:** Spark infiere los números JSON como `long` (64 bits). Los convertimos a `int`, suficiente para ids y años, para que coincidan con `juego_id` de `resultado` y ocupen menos memoria.

In [15]:
# 4.4 DataFrame juego
juego_raw = spark.read.option("multiLine", True).json(f"{DATA}/juegos.json")
print("Esquema original (inferido del JSON):")
juego_raw.printSchema()
juego_raw.show(3)

juego = juego_raw.select(
    F.col("juego_id").cast("int"),
    F.col("ano").alias("nombre_juego"),
    F.col("temporada").cast("int").alias("anio"),
    F.col("ciudad").alias("temporada"))

print(f"Filas del DataFrame juego: {juego.count()}  |  Años: "
      f"{juego.agg(F.min('anio')).first()[0]} a {juego.agg(F.max('anio')).first()[0]}\n")
juego.orderBy("anio").show(5)
juego.groupBy("temporada").count().show()
juego.printSchema()

Esquema original (inferido del JSON):
root
 |-- ano: string (nullable = true)
 |-- ciudad: string (nullable = true)
 |-- juego_id: long (nullable = true)
 |-- temporada: long (nullable = true)

+-----------+------+--------+---------+
|        ano|ciudad|juego_id|temporada|
+-----------+------+--------+---------+
|1896 Verano|Verano|       1|     1896|
|1900 Verano|Verano|       2|     1900|
|1904 Verano|Verano|       3|     1904|
+-----------+------+--------+---------+
only showing top 3 rows
Filas del DataFrame juego: 51  |  Años: 1896 a 2016

+--------+------------+----+---------+
|juego_id|nombre_juego|anio|temporada|
+--------+------------+----+---------+
|       1| 1896 Verano|1896|   Verano|
|       2| 1900 Verano|1900|   Verano|
|       3| 1904 Verano|1904|   Verano|
|       4| 1906 Verano|1906|   Verano|
|       5| 1908 Verano|1908|   Verano|
+--------+------------+----+---------+
only showing top 5 rows
+---------+-----+
|temporada|count|
+---------+-----+
|   Verano|   29|
| 

✅ **Resultado**
- El DataFrame `juego` tiene **51 ediciones**, desde **1896 hasta 2016**: 29 de verano y 22 de invierno. Incluye los Juegos Intercalados de **1906** (Atenas).
- `multiLine=True` permitió leer el arreglo JSON completo. El esquema original mostraba los nombres engañosos (`ano`, `ciudad`, `temporada`), que corregimos a **`nombre_juego`**, **`anio`** y **`temporada`** (Verano/Invierno).
- `juego_id` y `anio` pasaron de **long a integer**, el mismo tipo que `juego_id` en `resultado`, para el join.

### 4.5 DataFrame integrado: deportista + resultado + equipo + evento + juego

🎯 **Qué haremos**
- Partir de `resultado` (tabla puente) y unirla con:
  - `deportista` por `deportista_id`,
  - `equipo` por `equipo_id`,
  - `evento` por `evento_id`,
  - `juego` por `juego_id`.
- Ordenar las columnas de forma lógica, guardar el resultado en **caché** y revisar el **plan físico** que eligió Spark.

**Decisiones técnicas (optimización)**
1. **`left join` desde `resultado`:** conservamos **todas** las participaciones (271.116), aunque falte el dato en alguna tabla (por ejemplo, los 9 `evento_id` nulos). Un `inner join` las habría eliminado en silencio.
2. **`broadcast()` en `equipo`, `evento` y `juego`:** son tablas pequeñas (1.184, 765 y 51 filas). Spark envía **una copia completa a cada ejecutor** y hace el join localmente (**BroadcastHashJoin**), sin redistribuir (*shuffle*) las 271 mil filas de resultados. `deportista` (135 mil filas) es grande, así que se une con un join normal (**SortMergeJoin**).
3. **`cache()`:** el DataFrame integrado se usará en todas las secciones siguientes. Al guardarlo en memoria evitamos releer los archivos y repetir los 4 joins en cada acción.

In [16]:
# 4.5 DataFrame integrado
import io, contextlib

integrado = (resultado
    .join(deportista,          on="deportista_id", how="left")
    .join(F.broadcast(equipo), on="equipo_id",     how="left")
    .join(F.broadcast(evento), on="evento_id",     how="left")
    .join(F.broadcast(juego),  on="juego_id",      how="left")
    .select("resultado_id", "deportista_id", "nombre", "genero", "edad", "altura", "peso",
            "equipo_id", "equipo", "sigla", "evento_id", "evento", "deporte_id",
            "juego_id", "nombre_juego", "anio", "temporada", "medalla")
    .cache())

n_int = integrado.count()          # primera acción: ejecuta los joins y llena la caché
print(f"Filas resultado : {resultado.count():,}")
print(f"Filas integrado : {n_int:,}   (¿se conservaron todas?: {n_int == resultado.count()})")
print(f"Columnas        : {len(integrado.columns)}")
print(f"¿Está en caché? : {integrado.is_cached}\n")

# Estrategias de join elegidas por Spark (extraídas del plan físico)
buffer = io.StringIO()
with contextlib.redirect_stdout(buffer):
    integrado.explain()
joins = [l.strip().lstrip(":+- *()0123456789") for l in buffer.getvalue().splitlines() if "Join" in l]
print("Joins en el plan físico:")
for j in joins:
    print("  ", j[:90])

print("\nParticipaciones sin dato tras el join (nulos):")
for c in ["nombre", "equipo", "evento", "nombre_juego"]:
    print(f"  {c:13s}: {integrado.filter(F.col(c).isNull()).count()}")

integrado.select("nombre", "equipo", "sigla", "evento", "nombre_juego", "temporada", "medalla").show(5, truncate=False)

Filas resultado : 271,116
Filas integrado : 271,116   (¿se conservaron todas?: True)
Columnas        : 18
¿Está en caché? : True

Joins en el plan físico:
   BroadcastHashJoin [juego_id#172], [juego_id#279], LeftOuter, BuildRight, false
   BroadcastHashJoin [evento_id#196], [evento_id#96], LeftOuter, BuildRight, false
   BroadcastHashJoin [equipo_id#6], [equipo_id#126], LeftOuter, BuildRight, false
   SortMergeJoin [deportista_id#171], [deportista_id#0], LeftOuter
   BroadcastHashJoin [juego_id#172], [juego_id#279], LeftOuter, BuildRight, false
   BroadcastHashJoin [evento_id#196], [evento_id#96], LeftOuter, BuildRight, false
   BroadcastHashJoin [equipo_id#6], [equipo_id#126], LeftOuter, BuildRight, false
   SortMergeJoin [deportista_id#171], [deportista_id#0], LeftOuter

Participaciones sin dato tras el join (nulos):
  nombre       : 0
  equipo       : 200
  evento       : 9
  nombre_juego : 0
+---------------------------+-------+-----+--------------------------------------+---------

✅ **Resultado**
- El DataFrame `integrado` tiene **271.116 filas y 18 columnas**: exactamente las mismas filas que `resultado` (`True`). El `left join` no perdió ni duplicó ninguna participación.
- Está en **caché** (`is_cached = True`). El primer `count()` ejecutó los joins y guardó el resultado en memoria para las secciones siguientes.
- **Plan físico** (cada join aparece dos veces porque Spark muestra el plan final y el inicial de la ejecución adaptativa):

| Join | Estrategia | Por qué |
|---|---|---|
| `resultado` ↔ `deportista` | **SortMergeJoin** | ambas tablas son grandes (271 mil y 135 mil filas): se redistribuyen y ordenan por `deportista_id` |
| ↔ `equipo` | **BroadcastHashJoin** | 1.184 filas: se copia a cada ejecutor, sin *shuffle* |
| ↔ `evento` | **BroadcastHashJoin** | 765 filas |
| ↔ `juego` | **BroadcastHashJoin** | 51 filas |

- **Calidad de la integración:**
  - Todas las participaciones encontraron a su deportista (0 nulos en `nombre`) y su juego (0 nulos en `nombre_juego`).
  - **9** no tienen evento: son los 9 `evento_id = "#N/A"` originales.
  - **200** no tienen equipo: el `equipo_id` de esos deportistas no existe en `equipo.csv`. Gracias al `left join` esas filas se conservan con el resto de su información.

**Cierre de la sección 4:** creamos los DataFrames `evento` (765), `equipo` (1.184), `resultado` (271.116) y `juego` (51), cada uno con tipos correctos y nombres homogéneos, y los integramos con `deportista` en un solo DataFrame optimizado con **broadcast** y **cache**.